# SM00066 DBD position and length analysis (revised)

This notebook separates **protein-level** and **DBD-level** observations. Protein length is counted once per protein ID, whereas DBD position and length retain every domain observation. Absolute and relative DBD positions are shown separately.

In [ ]:
# Colab only, if the packages are unavailable:
# !pip -q install pandas numpy matplotlib scipy

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, MultipleLocator
from scipy.stats import gaussian_kde, pearsonr, spearmanr

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.bbox": "tight",
})

csv_path = Path("/content/aor0-5_SM00066_Protein_ID_Length.csv")
output_dir = Path(".")
outprefix = "SM00066"

## Load and validate the data

In [ ]:
required = ["ID", "Protein_Length", "DBD_Start", "DBD_End", "DBD_Length"]
df = pd.read_csv(csv_path)
missing_columns = sorted(set(required) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

numeric_columns = ["Protein_Length", "DBD_Start", "DBD_End", "DBD_Length"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

quality = pd.Series({
    "rows (DBD observations)": len(df),
    "unique protein IDs": df["ID"].nunique(dropna=True),
    "duplicate full rows": int(df.duplicated().sum()),
    "rows with missing required values": int(df[required].isna().any(axis=1).sum()),
    "start > end": int((df["DBD_Start"] > df["DBD_End"]).sum()),
    "DBD outside protein": int(((df["DBD_Start"] < 1) | (df["DBD_End"] > df["Protein_Length"])).sum()),
    "DBD length mismatch": int((df["DBD_Length"] != df["DBD_End"] - df["DBD_Start"] + 1).sum()),
})
display(quality.to_frame("value"))

invalid = (
    df[required].isna().any(axis=1)
    | (df["DBD_Start"] < 1)
    | (df["DBD_Start"] > df["DBD_End"])
    | (df["DBD_End"] > df["Protein_Length"])
    | (df["DBD_Length"] != df["DBD_End"] - df["DBD_Start"] + 1)
)
if invalid.any():
    raise ValueError(f"Found {invalid.sum()} invalid rows; inspect df.loc[invalid].")

# One row per protein for protein-level summaries.
# Retain all rows in df for domain-level summaries.
inconsistent_protein_lengths = df.groupby("ID")["Protein_Length"].nunique().gt(1)
if inconsistent_protein_lengths.any():
    bad_ids = inconsistent_protein_lengths[inconsistent_protein_lengths].index.tolist()
    raise ValueError(f"Protein_Length differs within ID: {bad_ids}")
protein_df = df.drop_duplicates("ID").copy()

df["DBD_Midpoint"] = (df["DBD_Start"] + df["DBD_End"]) / 2
df["Relative_DBD_Midpoint"] = df["DBD_Midpoint"] / df["Protein_Length"]
display(df[numeric_columns + ["Relative_DBD_Midpoint"]].describe().T)

## 1. Overlaid distributions with a common bin width

Protein length, DBD start, and DBD end are overlaid on one amino-acid axis. All three histograms use exactly the same bin edges and bin width, so their count heights are directly comparable. Protein length is counted once per protein; DBD coordinates are counted once per DBD observation.

In [ ]:
def plot_overlaid_distributions(protein_data, domain_data, bin_width=40, x_max=1600):
    # Use identical bin edges so that count heights are directly comparable.
    bin_edges = np.arange(0, x_max + bin_width, bin_width)
    grid = np.linspace(0, x_max, 800)
    series = [
        (protein_data["Protein_Length"].to_numpy(float), "Protein length", "#4C78A8"),
        (domain_data["DBD_Start"].to_numpy(float), "DBD start", "#F28E2B"),
        (domain_data["DBD_End"].to_numpy(float), "DBD end", "#59A14F"),
    ]
    fig, ax = plt.subplots(figsize=(12, 5))
    for values, label, color in series:
        ax.hist(values, bins=bin_edges, alpha=0.32, color=color, edgecolor=color,
                linewidth=0.8, label=f"{label} (n={len(values)})")
        if np.unique(values).size > 1:
            # Convert KDE density to expected count per common-width bin.
            kde_counts = gaussian_kde(values)(grid) * len(values) * bin_width
            ax.plot(grid, kde_counts, color=color, linewidth=2)
    ax.set_xlim(0, x_max)
    ax.set(xlabel="Amino acid position / Protein length (aa)", ylabel="Count",
           title=f"Protein length and DBD positions (common bin width = {bin_width} aa)")
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig

fig1 = plot_overlaid_distributions(protein_df, df, bin_width=40, x_max=1600)
plt.show()

## 2. Relative DBD position

The midpoint divided by protein length ranges from 0 (N terminus) to 1 (C terminus), allowing proteins of different lengths to be compared.

In [ ]:
def plot_relative_position(domain_data):
    values = domain_data["Relative_DBD_Midpoint"]
    bins = np.linspace(0, 1, 21)  # fixed width = 0.05
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(values, bins=bins, color="#59A14F", edgecolor="white")
    ax.axvline(values.median(), color="#D62728", linestyle="--", linewidth=1.5,
               label=f"Median = {values.median():.3f}")
    ax.set(xlim=(0, 1), xlabel="Relative DBD midpoint (0 = N terminus, 1 = C terminus)",
           ylabel="DBD count", title=f"Relative DBD position (n={len(values)} DBDs)")
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig

fig2 = plot_relative_position(df)
plt.show()

## 3. DBD length frequency

DBD length is a discrete integer variable, so exact-value counts are shown instead of a continuous KDE.

In [ ]:
def plot_dbd_length_counts(domain_data):
    counts = domain_data["DBD_Length"].astype(int).value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(counts.index, counts.values, width=0.8, color="#F28E2B", edgecolor="white")
    ax.set(xlabel="DBD length (aa)", ylabel="DBD count",
           title=f"Exact DBD length frequency (n={len(domain_data)} DBDs)")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    fig.tight_layout()
    return fig

fig3 = plot_dbd_length_counts(df)
plt.show()

## 4. Protein length versus DBD length

A deterministic vertical jitter reveals overlapping integer-valued DBD lengths. The unjittered values are used for Pearson and Spearman correlations. Correlation does not establish causation.

In [ ]:
def plot_protein_vs_dbd_length(domain_data, seed=20260529):
    x = domain_data["Protein_Length"].to_numpy(dtype=float)
    y = domain_data["DBD_Length"].to_numpy(dtype=float)
    pearson_r, pearson_p = pearsonr(x, y)
    spearman_rho, spearman_p = spearmanr(x, y)
    rng = np.random.default_rng(seed)
    y_jittered = y + rng.uniform(-0.18, 0.18, size=len(y))

    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.scatter(x, y_jittered, s=28, alpha=0.55, color="#4C78A8", edgecolors="none")
    stats_text = (f"Pearson r = {pearson_r:.3f} (p = {pearson_p:.3g})\n"
                  f"Spearman rho = {spearman_rho:.3f} (p = {spearman_p:.3g})")
    ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, ha="right", va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="0.8"))
    ax.set_xlim(0, 1600)
    ax.set_ylim(40, 65)
    ax.set(xlabel="Protein length (aa)", ylabel="DBD length (aa; jittered for display)",
           title=f"Protein length vs DBD length (n={len(domain_data)} DBDs)")
    ax.yaxis.set_major_locator(MultipleLocator(10))
    fig.tight_layout()
    return fig, pd.Series({
        "Pearson r": pearson_r, "Pearson p": pearson_p,
        "Spearman rho": spearman_rho, "Spearman p": spearman_p,
    })

fig4, correlation_results = plot_protein_vs_dbd_length(df)
display(correlation_results.to_frame("value"))
plt.show()

## Save publication-ready outputs

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
figures = {
    "overlaid_distributions": fig1,
    "relative_DBD_position": fig2,
    "DBD_length_frequency": fig3,
    "protein_vs_DBD_length": fig4,
}
for suffix, fig in figures.items():
    png_path = output_dir / f"{outprefix}_{suffix}.png"
    pdf_path = output_dir / f"{outprefix}_{suffix}.pdf"
    fig.savefig(png_path, dpi=300)
    fig.savefig(pdf_path)
    print(f"Saved: {png_path} and {pdf_path}")